# Four-model ink ensemble (PHerc0343 "First Letters" models) on our test surfaces

Models (Hugging Face, sha256-checked against `Nieuwlaar/pherc343-first-letters/ink/models.tsv`, in `models/villa_ensemble/`):
`dnative` (Nieuwlaar/ink9um-dense-native), `hecate` (scrollprize/hecate 9.6µm, run in bf16), `ink9-43` (scrollprize/ink_9um hybrid_3d2d seed 43) and a third member chosen with `SLOT3`: `v8in-1447` (YoussefMoNader/ink-8um-v8in-pherc1447-loo-w062, slowest; `ve.V8IN_STRIDE = 32` makes it 2.3x faster) or `ink9-42` (ink_9um seed 42, fast).

Run this notebook with your normal Python (see README / AGENT_INITIALIZATION.md); inference runs in the isolated villa env (`$VILLA_PY`, default `/opt/villa-venv`, villa @ `0fb38c45`) via `utils/villa_driver.py`.
Inputs: our 28-layer renders as uint8 zarr groups (`ves_zarrs_villa/<id>.zarr/0`). Ensemble = reference `ensemble_maps.py` (mean of per-model z-scores).

Per fragment, saved (not displayed) to `output/test_visualizations/villa_ensemble/`:
- `<name>_4models_d+0.png` — 2x2 grid of the four models, every panel at the render's native resolution
- `<name>_ensemble_d+0.png` — the ensemble at native resolution, with the composite view (max of layers 10–17, 1 cm bar) below

`render_depths(name)` writes a d = −5..+5 sweep to `<name>_depths/`. Run the sanity check (seg46527, PHerc0814) first: if ink is not visible, switch `DIRECTION` in cell 2.


In [ ]:
import os
import gc
import subprocess
import sys
import time

import numpy as np

VILLA_PY = os.getenv("VILLA_PY", "/opt/villa-venv/bin/python")
if os.path.abspath(sys.prefix) == os.path.dirname(os.path.dirname(os.path.abspath(VILLA_PY))):
    raise RuntimeError("select your normal Python kernel; the villa venv is only used internally for inference")

REPO = os.getenv("NT_REPO", os.getcwd())
os.chdir(REPO)
if REPO not in sys.path:
    sys.path.insert(0, REPO)
import importlib
from utils import villa_ensemble as ve
ve = importlib.reload(ve)

ZARR_DIR = os.getenv("VESUVIUS_ZARR_PATH", os.path.join(REPO, "ves_zarrs2"))
OUT_DIR = os.path.join(REPO, "output", "test_visualizations", "villa_ensemble")
# third ensemble member: "v8in-1447" (PHerc1447 fine-tune, slowest model) or "ink9-42" (ink_9um seed 42, fast)
SLOT3 = "v8in-1447"
ve.set_slot3(SLOT3)
# v8in tile stride: None = its default 21 (full quality); 32 = 2.3x faster, small fidelity loss
ve.V8IN_STRIDE = None
# depths d computed (stretch statistics pool over all of them); render_depths() sweeps -5..+5 instead
DEPTHS = (0,)
WRITE_DEPTH = 0
# layer order that reads the text side of our (training-oriented) renders: seg46527 ink 0.67 vs bg 0.14 forward, none reverse
DIRECTION = "forward"
COMPILE = False
TITLES = {"dnative": "dense_native (16k)", "hecate": "Hecate 9.6µm",
          "v8in-1447": "v8-in PHerc1447 LOO-w062", "ink9-42": "ink_9um seed 42", "ink9-43": "ink_9um seed 43"}

# name -> zarr id; mesh-rendered ids must also be in assemble_test_segments.FRAGMENTS
FRAGMENTS = {
    "auto_grown_20260814140748": 20260814140748,
    "auto_grown_20260717193517": 20260717193517,
    "auto_grown_20260720090842": 20260720090842,
    "auto_grown_20260723112922652_merged": 20260723112922,
    "auto_grown_20260921094413486": 20260921094413,
    "auto_grown_20260918132724424": 20260918132724,
    "auto_grown_20260922073234974": 20260922073234,
    "auto_grown_20260922161631422_merged": 20260922161631,
    "PHerc0800_z9264_w040_abf": 20260928000001,
    "PHerc0125_z15344_w040_abf": 20260928000002,
    "PHerc0211_z7312_w080_abf": 20260928000003,
    "PHerc0211_z5520_w040_abf": 20260928000004,
    "PHerc0813_z13088_w040_abf": 20260928000008,
    # sanity check: training fragment seg46527 (PHerc0814), native surface volume with known ink
    "seg46527_PHerc0814": 20260226000000,
}


def ensure_assembled(scroll_id):
    """render a test zarr from its tifxyz mesh when absent; the assembler writes the mask after the last layer"""
    zarr_path = os.path.join(ZARR_DIR, f"{scroll_id}.zarr")
    if scroll_id == "20260226000000":  # sanity segment: published surface volume, already in the models' format
        subprocess.run([sys.executable, os.path.join(REPO, "fetch_sanity_segment.py")], cwd=REPO, check=True)
        return
    mask_path = os.path.join(REPO, "masks", f"{scroll_id}.png")
    if not os.path.isdir(zarr_path):
        subprocess.run([sys.executable, os.path.join(REPO, "assemble_test_segments.py"), "--only", str(scroll_id),
                        "--out-dir", ZARR_DIR], cwd=REPO, check=True)
    zarray = os.path.join(zarr_path, ".zarray")
    if not (os.path.isfile(mask_path) and os.path.getmtime(mask_path) >= os.path.getmtime(zarray)):
        raise RuntimeError(f"{zarr_path} is still rendering (or was interrupted: rerun the assembler with --force --only {scroll_id})")


def render(name):
    """2x2 model grid + ensemble (with the composite view and a 1 cm bar below) at WRITE_DEPTH"""
    scroll_id = str(FRAGMENTS[name])
    t0 = time.time()
    ensure_assembled(scroll_id)
    ve.ensure_villa_input(scroll_id, ZARR_DIR)
    ve.run_models(scroll_id, DEPTHS, [DIRECTION], compile_model=COMPILE)
    print("[ensemble] stretching + z-score ensemble ...", flush=True)
    stretched, ensemble, stats = ve.ensemble(scroll_id, DEPTHS, DIRECTION, WRITE_DEPTH)
    tag = f"d{WRITE_DEPTH:+d}"
    grid_path = os.path.join(OUT_DIR, f"{name}_4models_{tag}.png")
    ens_path = os.path.join(OUT_DIR, f"{name}_ensemble_{tag}.png")
    ve.save_grid([(f"{TITLES[k]}  {tag} {DIRECTION}", stretched[k]) for k, _ in ve.MODELS], 2, grid_path)
    ve.save_ensemble_figure(scroll_id, ensemble, f"4-model ensemble (mean z-score)  {tag} {DIRECTION}", ens_path)
    del stretched, ensemble, stats
    gc.collect()
    print(f"[saved] {grid_path}\n[saved] {ens_path}\n[done] {name} in {time.time() - t0:.0f}s")


def render_depths(name, depths=tuple(range(-5, 6))):
    """depth sweep into OUT_DIR/<name>_depths/; stats pool over all depths, models skip depths they cannot reach"""
    scroll_id = str(FRAGMENTS[name])
    t0 = time.time()
    ensure_assembled(scroll_id)
    ve.ensure_villa_input(scroll_id, ZARR_DIR)
    ve.run_models(scroll_id, depths, [DIRECTION], compile_model=COMPILE)
    out_dir = os.path.join(OUT_DIR, f"{name}_depths")
    for d in depths:
        stretched, ensemble, stats = ve.ensemble(scroll_id, depths, DIRECTION, d)
        tag = f"d{d:+d}"
        used = [k for k, _ in ve.MODELS if not np.isnan(stretched[k]).all()]
        titles = [f"{TITLES[k]}  {tag} {DIRECTION}" + ("" if k in used else "  (no map: out of depth range)")
                  for k, _ in ve.MODELS]
        ve.save_grid([(title, stretched[k]) for title, (k, _) in zip(titles, ve.MODELS)], 2,
                     os.path.join(out_dir, f"{name}_4models_{tag}.png"))
        ve.save_ensemble_figure(scroll_id, ensemble, f"{len(used)}-model ensemble (mean z-score)  {tag} {DIRECTION}",
                                os.path.join(out_dir, f"{name}_ensemble_{tag}.png"))
        del stretched, ensemble, stats
        gc.collect()
        print(f"[saved] {tag}: {len(used)} models", flush=True)
    print(f"[done] {name} depths {depths[0]:+d}..{depths[-1]:+d} in {time.time() - t0:.0f}s -> {out_dir}")


def render_all(names=None):
    """overnight: render every fragment, logging failures instead of stopping; finished maps are cached"""
    failed = []
    for name in names or FRAGMENTS:
        try:
            render(name)
        except Exception as exc:
            failed.append(name)
            print(f"[FAILED] {name}: {exc}", flush=True)
    print(f"[render_all] {len(failed)} failed: {failed}")


print(f"models: {[short for short, _ in ve.MODELS]} | v8in stride: {ve.V8IN_STRIDE or 21} | villa python: {ve.VILLA_PY} | "
      f"out: {OUT_DIR} | depths={DEPTHS} direction={DIRECTION}")


In [ ]:
# --- sanity check: seg46527 (PHerc0814, 9.362µm), known ink ---
render("seg46527_PHerc0814")

In [ ]:
# --- test fragment 1: PHerc0813 ---
render("auto_grown_20260814140748")

In [ ]:
# --- test fragment 2: PHerc0211 merged ---
render("auto_grown_20260717193517")

In [ ]:
# --- test fragment 3: PHerc1203 ---
render("auto_grown_20260720090842")

In [ ]:
# --- test fragment 5: PHerc0826 merged ---
render("auto_grown_20260723112922652_merged")

In [ ]:
# --- test fragment 6: PHerc0846A ---
render("auto_grown_20260921094413486")

In [ ]:
# --- test fragment 7: PHerc0175A (8.64µm) ---
render("auto_grown_20260918132724424")

In [ ]:
# --- test fragment 8: PHerc0306B (8.64µm) ---
render("auto_grown_20260922073234974")

In [ ]:
# --- test fragment 9: PHerc0800 merged (8.64µm) ---
render("auto_grown_20260922161631422_merged")

In [ ]:
# --- test fragment 11: PHerc0800 expanded auto-winder segment (8.64µm) ---
render("PHerc0800_z9264_w040_abf")

In [ ]:
# --- test fragment 12: PHerc0125 expanded auto-winder segment ---
render("PHerc0125_z15344_w040_abf")

In [ ]:
# --- test fragment 13: PHerc0211 expanded auto-winder segment ---
render("PHerc0211_z7312_w080_abf")

In [ ]:
# --- PHerc0211 auto-winder segment z5520 ---
render("PHerc0211_z5520_w040_abf")

In [ ]:
# --- PHerc0813 auto-winder segment z13088 ---
render("PHerc0813_z13088_w040_abf")


In [ ]:
# --- optional depth sweep d = -5..+5 for one segment (v8-in only reaches -2..+2 on 28-layer renders) ---
# render_depths("PHerc0813_z13088_w040_abf")


In [ ]:
# --- overnight: every fragment (incl. sanity), continues past failures ---
render_all()
